# weight-decay-decoupled — faded example 1: Apply decoupled weight-decay before the Adam update

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`. Running the beacon reports progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: decoupled weight decay (AdamW)` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`weight-decay-decoupled`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "weight-decay-decoupled"
DD_SUBTOPIC = "Optimizer: decoupled weight decay (AdamW)"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In AdamW, weight decay is applied directly to the parameter with `p.mul_(1 - lr * wd)` BEFORE the Adam gradient update. This separates the regularization from the gradient — the decay shrinks the parameter regardless of the gradient's scale, which is the defining characteristic of 'decoupled' weight decay.

## Faded exercise 1

Implement `adamw_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step)`. The Adam moment update and bias-correction step are already filled in. Complete the blank that applies the decoupled weight decay to `p` BEFORE the Adam update.

Return nothing — `p`, `m`, `v` are mutated in place.

**Fill in:** Apply decoupled weight decay to p in-place: p.mul_(1 - lr * wd). This must happen before the Adam update.

In [ ]:
import torch as t

def adamw_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    # Decoupled weight decay (AdamW's defining step)
    raise NotImplementedError()  # TODO: Apply decoupled weight decay to p in-place: p.mul_(1 - lr * wd). This must happen before the Adam update.
    # Adam moment update
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    # Bias correction
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    # Adam update
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

# Exercise it
t.manual_seed(1)
p = t.tensor([0.6])
grad = t.tensor([0.05])
m = t.zeros(1); v = t.zeros(1)
adamw_one_step(p, grad, m, v, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01, step=1)
print('p:', p.item())


import torch as t

def adamw_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    p.mul_(1 - lr * wd)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

def _test():
    t.manual_seed(0)
    p = t.tensor([0.5])
    grad = t.tensor([0.1])
    m = t.zeros(1); v = t.zeros(1)
    lr, b1, b2, eps, wd = 1e-3, 0.9, 0.999, 1e-8, 0.01
    adamw_one_step(p, grad, m, v, lr, b1, b2, eps, wd, step=1)
    # Reference from torch.optim.AdamW
    ref_p = t.tensor([0.5], requires_grad=True)
    opt = t.optim.AdamW([ref_p], lr=lr, betas=(b1, b2), eps=eps, weight_decay=wd)
    opt.zero_grad()
    ref_p.grad = t.tensor([0.1])
    opt.step()
    assert t.allclose(p, ref_p.detach(), atol=1e-6), f'p={p.item()}, ref={ref_p.item()}'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def adamw_one_step(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    p.mul_(1 - lr * wd)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

t.manual_seed(1)
p = t.tensor([0.6])
grad = t.tensor([0.05])
m = t.zeros(1); v = t.zeros(1)
adamw_one_step(p, grad, m, v, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, wd=0.01, step=1)
print('p:', p.item())
```
</details>